# Introduction to Learning to Rank with Pyterrier

This notebook covers the basic steps to setup a Learning to Rank (LTR) pipeline with Pyterrier. After the setup, we have a quick look at "pipes" to understand what kind of features can be used in general to train LTR methods.

Afterward, we have a look at Pyterrier's support of common Machine Learning (ML) software frameworks - `scikit-learn`, `xgboost`, and `lightgbm`.

Finally, we have a look at how you can define custom features. LTR is not only about the ML methods, but also covers the feature engineering. For this reason, we will include the "number of authors" as an additional feature into our LTR pipeline. 

## Setup

Install Pyterrier.

In [ ]:
%pip install python-terrier

Imports.

In [ ]:
import numpy as np
import pandas as pd
import pyterrier as pt
import ir_datasets

Download and index the TREC COVID dataset, get topics and qrels.

In [ ]:
dataset = ir_datasets.load("cord19/trec-covid")

indexer = pt.IterDictIndexer(
    index_path="./index-cord19", # path to the index
    overwrite=True, 
    meta={"docno": 100, 
          "doi": 100, 
          "abstract": 20480,
          "title": 2048
          },
    meta_reverse=["docno"],
    text_attrs=["doi", "title", "abstract"],  
)

def document_generator():
    for doc in dataset.docs_iter():
        yield {
            "docno": doc.doc_id,
            "doi": doc.doi,
            "abstract": doc.abstract,
            "title": doc.title
        }

docs = document_generator()

indexer.index(docs)
index = pt.IndexFactory.of("./index-cord19")

topics = pd.DataFrame([
    {"qid": q.query_id, "query": q.title}
    for q in dataset.queries_iter()
])

topics = pd.DataFrame(dataset.queries)
topics.rename(columns={"query_id": "qid", "title": "query"}, inplace=True)  

qrels = pd.DataFrame(dataset.qrels)
qrels.rename(columns={"query_id": "qid", "doc_id": "docno", "relevance": "label"}, inplace=True)

## Quick intro to pipes.

Later, we setup a two-stage ranking pipeline with a BM25-based first-stage ranker whose outputs will be reranked by a ML method given the predefined features. In the example below, we setup three different batch retrievers.

In [ ]:
BM25 = pt.terrier.Retriever(index, controls = {"wmodel": "BM25"})
TF_IDF =  pt.terrier.Retriever(index, controls = {"wmodel": "TF_IDF"})
PL2 =  pt.terrier.Retriever(index, controls = {"wmodel": "PL2"})

We make a `pipe` by transforming the BM25 outputs with the help of PL2 and TFIDF (the latter two batch retrievers are used to generate our features).

In [ ]:
pipe = BM25 >> (TF_IDF ** PL2)

Let's use an example query and have a look at the outputs. The results are ranked by the BM25 score and the `feature` column contains a 1-D array with the features based on TFIDF and PL2. Later on, we use these BM25 candidates in combination with their feature representations.

In [ ]:
pipe.search("coronavirus immunity")

As an alternative to the previous code snippets, we can also create a `FeaturesBatchRetrieve` object.

In [ ]:
fbr = pt.terrier.FeaturesRetriever(index, controls = {"wmodel": "BM25"}, features=["WMODEL:TF_IDF", "WMODEL:PL2"]) 
(fbr % 5).search("coronavirus immunity")

## Learning-to-rank

Finally, we can have a look at the actual ML methods and Pyterrier's support of well-known ML software frameworks.

As it is crucial to avoid test/training data leakage, we have to split our topics for the training, validation, and testing. In this introduction, we do a simple "static" split. However, for more reliable evaluations you should consider [cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)), i.e., different test/training data splits.

In [ ]:
train_topics, validation_topics, test_topics = np.split(topics, [int(.6*len(topics)), int(.8*len(topics))])

Likewise, we have to split qrels for some ML methods.

In [ ]:
train_min = train_topics['qid'].astype(int).min()
train_max = train_topics['qid'].astype(int).max()
train_qrels = qrels[(qrels['qid'].astype(int) >= train_min) & (qrels['qid'].astype(int) <= train_max)]

val_min = validation_topics['qid'].astype(int).min()
val_max = validation_topics['qid'].astype(int).max()
validation_qrels = qrels[(qrels['qid'].astype(int) >= val_min) & (qrels['qid'].astype(int) <= val_max)]

test_min = test_topics['qid'].astype(int).min()
test_max = test_topics['qid'].astype(int).max()
test_qrels = qrels[(qrels['qid'].astype(int) >= test_min) & (qrels['qid'].astype(int) <= test_max)]

### `scikit-learn`

First of all, we have a look at how Pyterrier support [`scikit-learn`](https://scikit-learn.org/stable/). In the example below, we user Random Forest regression, Logistic regression, and Support Vector regression but the framework support more ML methods. Please have a look at the documentation.

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn import svm

# Create the regressor object.
rf = RandomForestRegressor(n_estimators=400)
# Pipe the outputs of the first stage ranking and the corresponding features into the regressor.
rf_pipe = fbr >> pt.ltr.apply_learned_model(rf)
# Fit the regressor with the given documents corresponding to the training topics.
rf_pipe.fit(train_topics, qrels)

# Logistic regression (default parametrization, have a look at the documentation to specify hyperparamters)
lr = LogisticRegression()
lr_pipe = fbr >> pt.ltr.apply_learned_model(lr)
lr_pipe.fit(train_topics, qrels)

# Support Vector regression (default parametrization, have a look at the documentation to specify hyperparamters)
svr = svm.SVR()
svr_pipe = fbr >> pt.ltr.apply_learned_model(svr)
svr_pipe.fit(train_topics, qrels)

# Determine the results with the help of the test topics
results = pt.Experiment([PL2, rf_pipe, lr_pipe, svr_pipe], test_topics, qrels, ["map"], names=["PL2 (Baseline)", "Random forest", "Logistic regression", "Support vector regression"])
results

### LambdaMART and Gradient Boosting with `xgboost` and `lightgbm`

[LambdaMART](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf) is a well-known and established LTR technique. It is implemented into both `xgboost` and `lightgbm` (both packages have to be installed on Codespaces). In the example below, it uses the nDCG measure in its objective function. Besides qrels for the training, it also requires qrels for the validation to optimize the learned weights. Apart from that, the implementation is similar to the previous example. For more details please have a look at the linked paper.


In [ ]:
%pip install xgboost lightgbm

In [ ]:
import xgboost as xgb

lmart_x = xgb.sklearn.XGBRanker(objective='rank:ndcg',
      learning_rate=0.1,
      gamma=1.0,
      min_child_weight=0.1,
      max_depth=6,
      verbose=2,
      random_state=42)

lmart_x_pipe = fbr >> pt.ltr.apply_learned_model(lmart_x, form="ltr")
lmart_x_pipe.fit(train_topics, train_qrels, validation_topics, validation_qrels)

import lightgbm as lgb

lmart_l = lgb.LGBMRanker(task="train",
    min_data_in_leaf=1,
    min_sum_hessian_in_leaf=100,
    max_bin=255,
    num_leaves=7,
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[1, 3, 5, 10],
    learning_rate= .1,
    importance_type="gain",
    num_iterations=10)
lmart_l_pipe = fbr >> pt.ltr.apply_learned_model(lmart_l, form="ltr")
lmart_l_pipe.fit(train_topics, train_qrels, validation_topics, validation_qrels)

pt.Experiment(
    [PL2, lmart_x_pipe, lmart_l_pipe],
    test_topics,
    test_qrels,
    ["map"],
    names=["PL2 Baseline", "LambdaMART (xgBoost)", "LambdaMART (LightGBM)" ]
)

## Custom features a.k.a. "feature engineering"

In the earlier example, we have used the scores of lexical-based matching functions. However, in our case, we might want to include other bibliometric or network-based metadata as additional features. In the following example, we include the "number of authors" as a LTR features.

First of all, we have to load the metadata file into a DataFrame.

In [ ]:
# '/root/.ir_datasets/cord19/2020-07-16/metadata.csv' on Google Colab
metadata = pd.read_csv('~/.ir_datasets/cord19/2020-07-16/metadata.csv', low_memory=False)

Afterward, we use the code form the previous notebook in a slightly modified way to the determine the raw author count that is added as an additional feature to the TFIDF and PL2 scores. If you do not want to use these features, simply use a `BatchRetriever` or modify the combination of features in `_features()`.

The output shows that there is an additional third feature that represents the number of authors.

In [ ]:
# for faster access write the author information into a dictionary
author_dict = {}
for id, authors in zip(metadata['cord_uid'], metadata['authors']):
  author_dict[id] = authors
author_dict

def authors(docno):
  
    raw_authors = author_dict[docno]
    if isinstance(raw_authors, str):
      authors = raw_authors.split(';')
      num_authors = len(authors)
      return num_authors

    return 1

def _features(row):
    f1 = authors(row["docno"])
    features = np.append(row['features'], np.array([f1]))
    return features

fbr = pt.terrier.FeaturesRetriever(index, controls = {"wmodel": "BM25"}, features=["WMODEL:TF_IDF", "WMODEL:PL2"]) 
# br = pt.BatchRetrieve(index, wmodel="BM25") >> pt.apply.doc_features(_features)

p = fbr >> pt.apply.doc_features(_features)
# p = br >> pt.apply.doc_features(_features)

p.search("coronavirus immunity")

Let's train and compare the same regressor with different feature sets.

In [ ]:
rf = RandomForestRegressor(n_estimators=400)
rf_pipe = fbr >> pt.ltr.apply_learned_model(rf)
rf_pipe.fit(train_topics, qrels)

rfa = RandomForestRegressor(n_estimators=400)
rfa_pipe = fbr >> pt.apply.doc_features(_features) >> pt.ltr.apply_learned_model(rfa)
rfa_pipe.fit(train_topics, qrels)

results = pt.Experiment([PL2, rf_pipe, rfa_pipe], test_topics, qrels, ["map"], names=["PL2 (Baseline)", "Random forest (without authors)", "Random forest (with authors)"])
results

Unfortunately, the "number of authors" features did not improve the retrieval performance. Now it is up to you to find some reasonable and useful features! :)

### Additional resources 

Pyterrier
- [Pyterrier documentation: Learning to Rank](https://pyterrier.readthedocs.io/en/latest/ltr.html#introduction)
- [Pyterrier documentation: apply.doc_features](https://pyterrier.readthedocs.io/en/latest/apply.html#pyterrier.apply.doc_features)
- [Pyterrier Notebook: Learning to Rank](https://colab.research.google.com/github/terrier-org/pyterrier/blob/master/examples/notebooks/ltr.ipynb#scrollTo=YTI_ax4K19nl)

Literature
- [Paper: From RankNet to LambdaRank to LambdaMART: An Overview](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf)
- [Wikipedia: Cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics))

Software
- [scikit-learn](https://scikit-learn.org/stable/)
- [lightgbm](https://lightgbm.readthedocs.io)
- [xgboost](https://xgboost.readthedocs.io/)
- [fastrank (another interesting LTR toolkit)](https://github.com/jjfiv/fastrank)